In [4]:
import requests
from bs4 import BeautifulSoup

#scraper
def scraper():
    #counter to parse through each page
    page_num = 1
    
    #temp data
    complete = False
    temp_l = []

    temp_stock = []
    temp_sale = []

    #connect to website
    while complete is False:
        ref_site = f"https://shop.boulderplanet.sg/collections/climbing-shoes/climbing-shoes?page={page_num}"
        response = requests.get(ref_site)


        #make response usable
        txt = response.text
        cont = response.content
        #print(txt)
        
        #cleaning via BeautifulSoup
        tidied = BeautifulSoup(cont, "html.parser")
        #print(tidied)
        
        shoe_list = tidied.find()

        shoe_cards = shoe_list.find_all("product-card")
        #print(shoe_cards)
        #print(len(shoe_cards))

        #check if there are more pages to parse
        
        #case 1: no more pages
        if len(shoe_cards) == 0:
            complete = True
        
        #case 2: more pages to parse
        else:
            for shoe in shoe_cards:
                #default assume in stock
                stock = 1
                sale = 0

                #obtain shoe name
                shoe_n = shoe.select_one("span.product-card__title").get_text(strip = True)

                #check if shoe is on sale, obtain either retail or sale price
                if shoe.select_one("sale-price.text-subdued") == None:
                    shoe_p = float(shoe.select_one("sale-price.text-on-sale").get_text(strip = True)[11:-4])
                else:
                    shoe_p = float(shoe.select_one("sale-price.text-subdued").get_text(strip = True)[11:-4])
                
                #check if shoe is out of stock/on sale
                prod_listing = shoe.select_one("div.product-card__badge-list")

                if prod_listing is not None and prod_listing.get_text(strip = True) == "Sold out":
                    stock = 0
                    temp_stock.append((shoe_n, shoe_p, stock, sale))

                elif prod_listing is not None and prod_listing.get_text(strip = True)[0:4] == "Save":
                    sale = 1
                    temp_sale.append((shoe_n, shoe_p, stock, sale))

                temp_l.append((shoe_n, shoe_p, stock, sale))   

            page_num += 1
    
    return (temp_l, temp_stock, temp_sale)

lst_items = scraper()[0]
lst_stock = scraper()[1]
lst_sale = scraper()[2]

print(lst_stock)
print(lst_sale)

[('Zenist Men’s', 210.0, 0, 0), ('2025 Defy LV', 150.0, 0, 0)]
[('Elektra Women', 97.5, 1, 1), ('Defy', 97.5, 1, 1), ('Zenist Women’s', 136.5, 1, 1), ('Shaman LV', 142.35, 1, 1), ('Geshido Velcro Women’s', 120.6, 1, 1), ('Kira', 96.0, 1, 1)]


In [ ]:
#db helper functions

import sqlite3

#getter function
def db_get(salestock, blean):
    conn = sqlite3.connect("shoebase.db")
    
    curs = conn.cursor()
    
    curs.execute(f"""
        SELECT "ShoeN", "ShoeP", "Stock", "Sale" FROM "ShoeD"
        WHERE {salestock} = ?
                """, (blean, ))

    lst_unstock = curs.fetchall()

    conn.commit()
    conn.close()

    return lst_unstock

#updater
def update(item, value, new_val):
    conn = sqlite3.connect("shoebase.db")
    
    curs = conn.cursor()

    curs.execute(f"""
                UPDATE "Spending"
                SET {value} = '{new_val}'
                WHERE "ShoeN" = ?
                 """, (item, ))

    conn.commit()
    conn.close()

db_nostock = db_get("Stock", 0)
db_onsale = db_get("Sale", 1)

print(db_nostock)
print(db_onsale)

[('Zenist Men’s', 210.0, 0, 0), ('2025 Defy LV', 150.0, 0, 0)]
[('Elektra Women', 97.5, 1, 1), ('Defy', 97.5, 1, 1), ('Zenist Women’s', 136.5, 1, 1), ('Shaman LV', 142.35, 1, 1), ('Geshido Velcro Women’s', 120.6, 1, 1), ('Kira', 96.0, 1, 1)]


In [ ]:
#updater

#find the symmetric difference (items to be updated)

#test values stock
test1 = [('Zenist Men’s', 210.0, 0, 0), ('testshoex', 1.00, 0, 0), ('testshoey', 2.00, 0, 0), ('2025 Defy LV', 150.0, 0, 0)] #"stored" data set
test2 = [('2025 Defy LV', 150.0, 0, 0), ('testshoe1', 1.00, 0, 0), ('testshoey', 2.00, 0, 0), ('testshoe2', 1.00, 0, 0), ('Zenist Men’s', 210.0, 0, 0)] #new data set

#test values sale
test3 = [('Zenist Men’s', 210.0, 1, 1), ('testshoex', 1.00, 1, 1), ('testshoey', 2.00, 1, 1), ('2025 Defy LV', 150.0, 1, 1)] #"stored" data set
test4 = [('2025 Defy LV', 150.0, 1, 1), ('testshoe1', 1.00, 1, 0), ('testshoey', 2.00, 1, 1), ('testshoe2', 1.00, 1, 1), ('Zenist Men’s', 210.0, 1, 1)] #new data set

def symm_difference(rec, upd):
    symDiff = set(i[0] for i in rec) ^ set(i[0] for i in upd)
    #first array to signal from on sale/out of stock -> off sale/in stock
    #second array to signal from off sale/in stock -> on sale/out of stock
    return [[i for i in rec if i[0] in symDiff], [i for i in upd if i[0] in symDiff]]

#stock updater
def stock_update():
    pass

#sale updater
def sale_update():
    pass

updates_stock = symm_difference(test1, test2)
updates_sale = symm_difference(test3, test4)

print(updates_stock)
print(updates_sale)

[[('testshoex', 1.0, 0, 0)], [('testshoe1', 1.0, 0, 0), ('testshoe2', 1.0, 0, 0)]]
[[('testshoex', 1.0, 1, 1)], [('testshoe1', 1.0, 1, 0), ('testshoe2', 1.0, 1, 1)]]
